Import the module:

In [1]:
import asyncpg
import aio_pika
import documents
from api_client import utility_client

Create connections and docs instance:

In [7]:
rabbit_conn = await aio_pika.connect_robust(
    host="51.250.44.130",
    port=50602,
    login="knowledge-base-service",
    password="<your RABBITMQ_PASSWORD>",
)
channel = await rabbit_conn.channel()
util_client = utility_client.Client(channel, {"consuming_log": True, "llm": "openai"})

db_pool = await asyncpg.create_pool(
    host="51.250.44.130",
    port=17939,
    user="knowledge_base_service",
    password="<your PGPASSWORD>",
    database="knowledge_base_backend_db",
)

docs = documents.Documents(util_client, db_pool)

Working with documents:

In [8]:
created_docs = await docs.create([{"title": "foo", "text": "bar"}])
print("created_docs:", created_docs)
await docs.add_category("dog")
categories = await docs.get_categories()
print("categories:", categories)

created_docs: [2]
categories: [{'id': 1, 'title': 'dog'}]


In [10]:
await docs.update(lambda d: {**d, "categories": [1]}, [1])
await docs.get_by_category_ids(required=[1])

[{'id': 1,
  'title': 'foo',
  'text': 'bar',
  'action_url': None,
  'source_url': None,
  'categoiries': [1]}]

Preprocessing:

In [11]:
await docs.run_search_preprocessing()

Starting search preprocessing tasks
Start preprocessing for vector search
Last preprocessing was 1970-01-01 03:00:00
1 docs found for preprocessing
Processing 1 docs...
[add_task_to_queue] response: [[0.05009923502802849, 0.006838448345661163, -0.02657940238714218, -0.03921975567936897, 0.042138777673244476, -0.02690529264509678, -0.0039674583822488785, 0.05268268287181854, 0.04582275450229645, -0.037655215710401535, 0.0371595099568367, 0.045573800802230835, -0.05296863242983818, -0.024013137444853783, -0.014238632284104824, -0.03336985781788826, 0.006151861976832151, 0.026213141158223152, 0.0161182451993227, -0.007415687665343285, 0.03760838881134987, -0.004227359313517809, -0.02452474646270275, -0.04898554086685181, -0.016754310578107834, -0.0225741695612669, -0.027964038774371147, -0.041229479014873505, -0.019826022908091545, -0.03284258022904396, -0.010579495690762997, 0.010401451028883457, -0.05048644542694092, -0.04684676229953766, -3.071356695727445e-05, 0.03029717691242695, 0.0

Cleanup:

In [14]:
await docs.delete_preprocessings([1])
await docs.delete([1])
await docs.delete_categories([1])

Close connections:

In [15]:
await rabbit_conn.close()
await db_pool.close()